In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('amazonproducts.csv', sep=';', on_bad_lines='skip')
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
df.head(3)

In [ ]:
print("PREPROCESAMIENTO: ANALISIS DE DATOS NULOS")
print("=" * 55)

columnas_interes = ['final_price', 'initial_price', 'images_count', 'video_count',
                    'reviews_count', 'root_bs_rank', 'rating', 'discount',
                    'item_weight', 'brand', 'categories']

nulos = df[columnas_interes].isnull().sum()
total = len(df)
porcentaje = (nulos / total * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Nulos': nulos,
    'No Nulos': total - nulos,
    '% Nulos': porcentaje
}).sort_values('% Nulos', ascending=False)

print(resumen_nulos)
print(f"\nTotal filas: {total}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colores = ['#e74c3c' if p > 30 else '#f39c12' if p > 10 else '#2ecc71' for p in porcentaje]
axes[0].barh(resumen_nulos.index, resumen_nulos['% Nulos'], color=colores)
axes[0].set_xlabel('% Nulos')
axes[0].set_title('Porcentaje de Nulos por Columna')
for i, v in enumerate(resumen_nulos['% Nulos']):
    axes[0].text(v + 0.5, i, f'{v}%', va='center', fontsize=9)

no_nulo_cols = ['final_price', 'images_count', 'video_count', 'reviews_count',
                'root_bs_rank', 'rating', 'brand', 'categories']
filas_completas = df[no_nulo_cols].dropna().shape[0]
filas_incompletas = total - filas_completas
axes[1].pie([filas_completas, filas_incompletas],
            labels=['Completas', 'Con nulos'], autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Filas completas vs incompletas (cols principales)')

plt.tight_layout()
plt.show()

In [ ]:
print("LIMPIEZA Y TRANSFORMACION DE COLUMNAS")
print("=" * 55)

def limpiar_precio(val):
    if pd.isna(val) or str(val).lower() == 'null':
        return np.nan
    val = str(val).replace('"', '').strip()
    if 'E+' in val or 'e+' in val:
        return np.nan
    val = val.replace(',', '.')
    try:
        r = float(val)
        return r if r < 5000 else np.nan
    except:
        return np.nan

def limpiar_descuento(val):
    if pd.isna(val):
        return 0
    val = str(val).replace('%', '').replace('-', '').strip()
    try:
        return abs(float(val))
    except:
        return 0

def parsear_peso(val):
    if pd.isna(val):
        return np.nan
    val = str(val).lower().strip()
    try:
        if 'pound' in val:
            return float(val.split()[0]) * 0.4536
        elif 'ounce' in val:
            return float(val.split()[0]) * 0.02835
        elif 'kilogram' in val:
            return float(val.split()[0])
        elif 'kg' in val:
            return float(val.replace('kg', '').strip())
        elif 'g' in val:
            num = float(val.replace('g', '').strip())
            return num / 1000 if num > 50 else num
        else:
            return float(val)
    except:
        return np.nan

def extraer_categoria(val):
    if pd.isna(val):
        return np.nan
    val = str(val).replace('[', '').replace(']', '').replace('"', '')
    partes = val.split(',')
    return partes[0].strip() if partes else np.nan

df['precio'] = df['final_price'].apply(limpiar_precio)
df['imagenes'] = pd.to_numeric(df['images_count'], errors='coerce')
df['videos'] = pd.to_numeric(df['video_count'], errors='coerce')
df['resenas'] = pd.to_numeric(df['reviews_count'], errors='coerce')
df['ranking'] = pd.to_numeric(df['root_bs_rank'], errors='coerce')
df['rating_num'] = pd.to_numeric(df['rating'], errors='coerce')
df['descuento'] = df['discount'].apply(limpiar_descuento)
df['peso_kg'] = df['item_weight'].apply(parsear_peso)
df['categoria'] = df['categories'].apply(extraer_categoria)

cols_procesadas = ['precio', 'imagenes', 'videos', 'resenas', 'ranking',
                   'rating_num', 'descuento', 'peso_kg', 'categoria']

nulos_post = df[cols_procesadas].isnull().sum()
porcentaje_post = (nulos_post / len(df) * 100).round(2)

resumen_post = pd.DataFrame({
    'Nulos': nulos_post,
    'No Nulos': len(df) - nulos_post,
    '% Nulos': porcentaje_post
}).sort_values('% Nulos', ascending=False)

print(resumen_post)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colores_post = ['#e74c3c' if p > 30 else '#f39c12' if p > 10 else '#2ecc71' for p in porcentaje_post]
axes[0].barh(resumen_post.index, resumen_post['% Nulos'], color=colores_post)
axes[0].set_xlabel('% Nulos')
axes[0].set_title('Nulos DESPUES de limpieza')
for i, v in enumerate(resumen_post['% Nulos']):
    axes[0].text(v + 0.5, i, f'{v}%', va='center', fontsize=9)

axes[1].bar(cols_procesadas, df[cols_procesadas].notna().sum(), color='#3498db')
axes[1].set_ylabel('Registros validos')
axes[1].set_title('Registros validos por columna procesada')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("MODELO 1: EL DETECTOR DE BEST SELLERS")
print("=" * 60)

df1 = df[['precio', 'imagenes', 'videos', 'resenas', 'ranking']].dropna()
print(f"Filas usadas (sin nulos): {len(df1)} de {len(df)}")
mediana_rank = df1['ranking'].median()
df1 = df1.copy()
df1['es_top'] = (df1['ranking'] <= mediana_rank).astype(int)

X1 = df1[['precio', 'imagenes', 'videos', 'resenas']]
y1 = df1['es_top']

X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.3, random_state=42)

arbol1 = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol1.fit(X1_train, y1_train)

y1_pred = arbol1.predict(X1_test)
print(f"\nPrecision del modelo: {accuracy_score(y1_test, y1_pred):.4f}")
print(classification_report(y1_test, y1_pred, target_names=['No Top', 'Top']))

nombre_features = ['precio', 'imagenes', 'videos', 'resenas']
print(f"Variable en la RAIZ del arbol: {nombre_features[arbol1.tree_.feature[0]]}")
print(f"Umbral de division: {arbol1.tree_.threshold[0]:.2f}")

print(f"\nImportancia de cada variable:")
for nombre, imp in zip(nombre_features, arbol1.feature_importances_):
    print(f"  {nombre}: {imp:.4f}")

In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(arbol1, feature_names=['precio', 'imagenes', 'videos', 'resenas'],
          class_names=['No Top', 'Top'], filled=True, rounded=True, fontsize=8)
plt.title('Modelo 1: Detector de Best Sellers', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("HIPOTESIS: Producto con 10 fotos y 0 videos")
print("-" * 45)

hipotesis = pd.DataFrame({
    'precio': [25.0],
    'imagenes': [10],
    'videos': [0],
    'resenas': [500]
})

pred1 = arbol1.predict(hipotesis)
prob1 = arbol1.predict_proba(hipotesis)

print(f"Prediccion: {'Top Seller' if pred1[0] == 1 else 'No Top'}")
print(f"Probabilidad de ser Top: {prob1[0][1]:.2%}")

print(f"\nRECOMENDACION PARA VENDEDOR CON RANKING BAJO:")
print(f"Basado en la importancia de variables del modelo:")
for nombre, imp in sorted(zip(nombre_features, arbol1.feature_importances_), key=lambda x: -x[1]):
    print(f"  -> Optimizar '{nombre}' (importancia: {imp:.4f})")

In [ ]:
print("=" * 60)
print("MODELO 2: EL PREDICTOR DE CALIDAD PREMIUM")
print("=" * 60)

df2 = df[['brand', 'precio', 'descuento', 'rating_num', 'resenas']].dropna()
df2 = df2[df2['brand'].notna() & (df2['brand'].str.strip() != '')].copy()
print(f"Filas usadas (sin nulos): {len(df2)} de {len(df)}")

le_marca = LabelEncoder()
df2['marca_cod'] = le_marca.fit_transform(df2['brand'])
df2['es_premium'] = (df2['rating_num'] >= 4.5).astype(int)

X2 = df2[['marca_cod', 'precio', 'descuento', 'resenas']]
y2 = df2['es_premium']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42)

arbol2 = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol2.fit(X2_train, y2_train)

y2_pred = arbol2.predict(X2_test)
print(f"\nPrecision del modelo: {accuracy_score(y2_test, y2_pred):.4f}")
print(classification_report(y2_test, y2_pred, target_names=['No Premium', 'Premium']))

print("Top 10 marcas con mas productos Premium:")
print(df2[df2['es_premium'] == 1]['brand'].value_counts().head(10))

In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(arbol2, feature_names=['marca_cod', 'precio', 'descuento', 'resenas'],
          class_names=['No Premium', 'Premium'], filled=True, rounded=True, fontsize=8)
plt.title('Modelo 2: Predictor de Calidad Premium', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("PRECIO VS FELICIDAD: Productos baratos con rating alto")
print("-" * 50)
baratos_buenos = df2[(df2['precio'] < df2['precio'].quantile(0.25)) & (df2['rating_num'] > 4.5)]
print(f"Productos baratos (Q1) con rating > 4.5: {len(baratos_buenos)}")
print(f"Descuento promedio: {baratos_buenos['descuento'].mean():.1f}%")
print(f"Resenas promedio: {baratos_buenos['resenas'].mean():.0f}")
print(f"\nMarcas de estos productos:")
print(baratos_buenos['brand'].value_counts().head(5))

print("\n" + "=" * 50)
print("PRUEBA DE ROBUSTEZ: Marca desconocida")
print("-" * 50)
print(f"Total marcas conocidas: {len(le_marca.classes_)}")

marca_nueva = "MarcaInventada2024"
try:
    le_marca.transform([marca_nueva])
except ValueError as e:
    print(f"ERROR al codificar '{marca_nueva}': {e}")

print(f"\nSOLUCION: Usar try/except con valor por defecto para marcas nuevas")

def predecir_con_marca(marca, precio, descuento, resenas):
    try:
        codigo = le_marca.transform([marca])[0]
    except ValueError:
        codigo = -1
    datos = pd.DataFrame({'marca_cod': [codigo], 'precio': [precio], 'descuento': [descuento], 'resenas': [resenas]})
    pred = arbol2.predict(datos)
    prob = arbol2.predict_proba(datos)
    return 'Premium' if pred[0] == 1 else 'No Premium', prob[0]

resultado, probabilidades = predecir_con_marca("MarcaInventada2024", 50.0, 10, 200)
print(f"Prediccion marca desconocida: {resultado}")
print(f"Probabilidades [No Premium, Premium]: {probabilidades}")

In [ ]:
print("=" * 60)
print("MODELO 3: EL CLASIFICADOR DE CATEGORIAS")
print("=" * 60)

df3 = df[['peso_kg', 'precio', 'rating_num', 'imagenes', 'categoria']].dropna()
print(f"Filas usadas (sin nulos): {len(df3)} de {len(df)}")
top_cats = df3['categoria'].value_counts().head(6).index.tolist()
df3 = df3[df3['categoria'].isin(top_cats)].copy()

le_cat = LabelEncoder()
df3['cat_cod'] = le_cat.fit_transform(df3['categoria'])

X3 = df3[['peso_kg', 'precio', 'rating_num', 'imagenes']]
y3 = df3['cat_cod']

X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.3, random_state=42)

arbol3 = DecisionTreeClassifier(max_depth=5, random_state=42)
arbol3.fit(X3_train, y3_train)

y3_pred = arbol3.predict(X3_test)
print(f"\nPrecision del modelo: {accuracy_score(y3_test, y3_pred):.4f}")
print(classification_report(y3_test, y3_pred, target_names=le_cat.classes_))

print(f"Categorias del modelo: {list(le_cat.classes_)}")
print(f"Total categorias: {len(le_cat.classes_)} (vs 2 clases en modelos anteriores)")
print(f"Esto explica la menor confianza: mas clases = mas dificil clasificar")

In [ ]:
plt.figure(figsize=(30, 14))
plot_tree(arbol3, feature_names=['peso_kg', 'precio', 'rating', 'imagenes'],
          class_names=list(le_cat.classes_), filled=True, rounded=True, fontsize=7)
plt.title('Modelo 3: Clasificador de Categorias', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_estimator(arbol3, X3_test, y3_test,
                                      display_labels=le_cat.classes_, ax=ax, cmap='Blues')
plt.title('Matriz de Confusion - Modelo 3', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
print("DESAFIO DEL PESO: Producto con peso = 0.0 kg")
print("-" * 50)

peso_cero = pd.DataFrame({'peso_kg': [0.0], 'precio': [20.0], 'rating_num': [4.0], 'imagenes': [3]})
pred_cat = arbol3.predict(peso_cero)
prob_cat = arbol3.predict_proba(peso_cero)

print(f"Peso = 0.0 kg -> Categoria predicha: {le_cat.inverse_transform(pred_cat)[0]}")
print(f"\nProbabilidades por categoria:")
for cat, prob in sorted(zip(le_cat.classes_, prob_cat[0]), key=lambda x: -x[1]):
    print(f"  {cat}: {prob:.2%}")

print(f"\nRangos de peso usados por el arbol:")
tree = arbol3.tree_
feat_names = ['peso_kg', 'precio', 'rating_num', 'imagenes']
for i in range(tree.node_count):
    if tree.feature[i] == 0:
        print(f"  Nodo {i}: peso_kg <= {tree.threshold[i]:.4f} kg")